# Explainable Financial Fraud Detection using Graph Neural Networks

## 1. Project Overview

Fraudulent transactions are rare and often involve relationships between accounts. This project represents transactions as a graph (accounts are nodes, transactions are directed edges) and compares graph models (**GCN** and **GAT**) against transaction-level baselines (Random Forest). Predictions are explained with **SHAP** and surfaced in a **Streamlit** risk dashboard.

**Dataset:** PaySim, a *synthetic* mobile-money transaction log (6.36M transactions, about 0.129% fraud). Findings reflect the simulator and may not transfer to real banking data.

### Objectives

- Analyse fraud patterns, including the structure of the account graph.
- Engineer transaction features, separating "known at scoring time" from "post-transaction" features.
- Build an account-to-account transaction graph.
- Train GCN and GAT edge classifiers with an imbalance-aware loss and compare them with baselines.
- Evaluate on a chronological test set with fraud-focused metrics (PR-AUC, precision, recall, top-k alert quality).
- Explain predictions with SHAP (tabular model and graph model).
- Deliver a Streamlit dashboard for risk monitoring and investigation.

### Notebook flow

| # | Section | # | Section |
| --- | --- | --- | --- |
| 1 | Overview | 7 | Chronological train / validation / test split |
| 2 | Setup | 8 | Graph windows and model definitions |
| 3 | Data loading | 9 | Training (RF, GCN, GAT) |
| 4 | EDA (incl. graph EDA) | 10 | Evaluation and sanity checks |
| 5 | Feature engineering | 11 | SHAP explainability |
| 6 | Graph construction | 12 | Save artifacts and Streamlit dashboard |

---

## 2. Setup and Configuration

All imports, seeds, paths and hyper-parameters live here so the rest of the notebook has no hidden state.

**Install once** (put this in `requirements.txt`, do not run it inside the notebook):

```
pandas numpy scipy scikit-learn matplotlib networkx joblib
torch torch_geometric shap streamlit
```

Run the notebook with **Kernel → Restart & Run All** so every output matches the code.

In [ ]:
import copy
import gc
import json
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GATConv

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.dpi"] = 100

# ---- Reproducibility ----
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Paths ----
DATA_PATH = Path("../data/PS_20174392719_1491204439457_log.csv")
MODEL_DIR = Path("saved_models")
MODEL_DIR.mkdir(exist_ok=True)

# ---- Data / split configuration ----
SAMPLE_FRAC = 1.0          # set to e.g. 0.1 for a quick debug run on 10% of rows
TEST_QUANTILE = 0.80       # last 20% of time steps = test
VAL_QUANTILE = 0.85        # within the earlier period, last 15% of steps = validation
WINDOW_STEPS = 24          # one graph per 24-hour window of transactions
TRAIN_LEGIT_FRAC = 0.10    # keep all training fraud, 10% of training legitimate rows

# ---- Model / training configuration ----
HIDDEN_DIM = 64
GAT_HEADS = 4
DROPOUT = 0.3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 30
PATIENCE = 5

print("Setup complete. Device:", DEVICE)


---

## 3. Data Loading and Quality Checks

The PaySim log is loaded and checked for missing values, duplicates and data types. Rows are sorted by `step` (hour) so that every later time-based split is correct.

In [ ]:
df = pd.read_csv(DATA_PATH)

if SAMPLE_FRAC < 1.0:
    df = df.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)

df = df.sort_values("step", kind="stable").reset_index(drop=True)

print(f"Shape:            {df.shape}")
print(f"Memory:           {df.memory_usage(deep=True).sum() / 1e6:,.0f} MB")
print(f"Missing values:   {int(df.isna().sum().sum())}")
print(f"Duplicate rows:   {int(df.duplicated().sum())}")
print(f"Time steps:       {int(df['step'].min())} to {int(df['step'].max())}")
print("\nColumn types:")
print(df.dtypes.to_string())
df.head()


---

## 4. Exploratory Data Analysis

### 4.1 Class Imbalance

In [ ]:
class_counts = df["isFraud"].value_counts()
class_pct = df["isFraud"].value_counts(normalize=True) * 100
print(pd.DataFrame({"count": class_counts, "percent": class_pct.round(4)}))


**Finding:** fraud is a tiny fraction of all transactions, so accuracy is meaningless. Evaluation uses precision, recall, F1 and PR-AUC.

### 4.2 Fraud by Transaction Type

In [ ]:
fraud_by_type = (
    df.groupby("type")["isFraud"]
    .agg(transaction_count="count", fraud_count="sum")
    .assign(fraud_rate_pct=lambda t: t["fraud_count"] / t["transaction_count"] * 100)
    .sort_values("fraud_rate_pct", ascending=False)
)
display(fraud_by_type)

fraud_by_type["fraud_rate_pct"].plot(kind="bar", figsize=(7, 4), title="Fraud Rate by Transaction Type")
plt.ylabel("Fraud rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


**Finding:** fraud occurs only in TRANSFER and CASH\_OUT. Transaction type alone is not enough, which motivates amount, balance and relational features.

### 4.3 Transaction Amount by Class

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for label, name in [(0, "Legitimate"), (1, "Fraud")]:
    ax.hist(np.log10(df.loc[df["isFraud"] == label, "amount"] + 1),
            bins=60, alpha=0.6, density=True, label=name)
ax.set_xlabel("log10(amount + 1)")
ax.set_ylabel("Density")
ax.set_title("Transaction Amount Distribution by Class")
ax.legend()
plt.tight_layout()
plt.show()

print(df.groupby("isFraud")["amount"].describe().round(0).to_string())


### 4.4 Fraud over Time

`step` is an hourly time step. The chronological split in Section 7 depends on how volume and fraud change over time.

In [ ]:
by_step = df.groupby("step")["isFraud"].agg(transactions="count", frauds="sum")
by_step["fraud_rate_pct"] = by_step["frauds"] / by_step["transactions"] * 100

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
by_step["transactions"].plot(ax=axes[0], title="Transactions per step")
by_step["fraud_rate_pct"].plot(ax=axes[1], title="Fraud rate (%) per step")
axes[1].set_xlabel("step (hour)")
plt.tight_layout()
plt.show()


### 4.5 Account-Balance Patterns in Fraud

Does fraud follow a recognisable pattern, such as draining the origin account? What do fraud destinations look like (merchant or customer)?

In [ ]:
fraud_df = df[df["isFraud"] == 1]

drained = (fraud_df["newbalanceOrig"] == 0) & (fraud_df["oldbalanceOrg"] > 0)
full_amount = np.isclose(fraud_df["amount"], fraud_df["oldbalanceOrg"]) & (fraud_df["oldbalanceOrg"] > 0)
print(f"Fraud where the origin account is drained to 0:  {drained.mean() * 100:.1f}%")
print(f"Fraud where amount equals the old balance:       {full_amount.mean() * 100:.1f}%")

legit_df = df[df["isFraud"] == 0]
legit_drained = (legit_df["newbalanceOrig"] == 0) & (legit_df["oldbalanceOrg"] > 0)
print(f"Legitimate rows with a drained origin account:   {legit_drained.mean() * 100:.1f}%")

dest_is_merchant = df["nameDest"].str.startswith("M")
print(f"\nMerchant destinations overall:   {dest_is_merchant.mean() * 100:.1f}% of transactions")
print(f"Fraud with a merchant destination: {int(df.loc[dest_is_merchant, 'isFraud'].sum())}")

print("\nisFlaggedFraud (built-in rule) vs isFraud:")
print(pd.crosstab(df["isFlaggedFraud"], df["isFraud"]).to_string())


**Reading this output:** if most fraud drains the origin account while few legitimate rows do, balance-based features will be extremely predictive. That is a property of PaySim's simulator, so Section 10 re-checks how much performance depends on those post-transaction features. The built-in `isFlaggedFraud` rule catches very little and is excluded from modelling.

### 4.6 Graph Structure EDA

Before building a GNN, check whether the account graph has relational structure to learn from.

In [ ]:
orig_counts = df["nameOrig"].value_counts()
dest_counts = df["nameDest"].value_counts()

print(f"Unique origin accounts:        {len(orig_counts):,}")
print(f"  appearing exactly once:      {(orig_counts == 1).mean() * 100:.1f}%")
print(f"Unique destination accounts:   {len(dest_counts):,}")
print(f"  appearing exactly once:      {(dest_counts == 1).mean() * 100:.1f}%")

both = orig_counts.index.intersection(dest_counts.index)
print(f"\nAccounts that are both origin and destination: {len(both):,}")

print("\nDestination in-degree (transactions received):")
print(dest_counts.describe().round(2).to_string())

fraud_dest_counts = dest_counts.loc[fraud_df["nameDest"].unique()]
print(f"\nFraud destination accounts:     {len(fraud_dest_counts):,}")
print(f"  receiving more than one transaction: {(fraud_dest_counts > 1).mean() * 100:.1f}%")

del orig_counts, dest_counts, both, fraud_dest_counts


**Why this matters:** a GNN only helps if accounts repeat. If most origin accounts appear once and accounts rarely act as both origin and destination, the graph is mostly isolated edges and star shapes around destination accounts. That is a limitation of PaySim, and it is discussed again when the GNNs are compared with the baseline in Section 10.

---

## 5. Feature Engineering

Features are logic shared with the Streamlit app, so they are written to `fraud_features.py` and imported. This keeps a single source of truth for the notebook and the dashboard.

Features are split into two groups:

- **Pre-transaction features:** known when a transaction is submitted (type, amount, old balances, hour of day, merchant flag, amount relative to old balances).
- **Post-transaction features:** use `newbalanceOrig` and `newbalanceDest` (and values derived from them). A real-time system may not have these at scoring time, so Section 10 measures how much performance depends on them.

Heavy-tailed monetary values are log-transformed for the neural networks; tree models use the raw values.

In [ ]:
%%writefile fraud_features.py
"""Shared feature engineering for the fraud notebook and the Streamlit dashboard."""
import numpy as np
import pandas as pd

TYPES = ["CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"]
TYPE_COLS = [f"type_{t}" for t in TYPES]
MONEY_COLS = ["amount", "oldbalanceOrg", "oldbalanceDest", "newbalanceOrig", "newbalanceDest"]

PRE_TXN_FEATURES = [
    "amount", "oldbalanceOrg", "oldbalanceDest",
    "amount_to_orig_balance", "amount_to_dest_balance",
    "dest_is_merchant", "hour_of_day",
] + TYPE_COLS

POST_TXN_FEATURES = [
    "newbalanceOrig", "newbalanceDest",
    "orig_balance_error", "dest_balance_error",
    "orig_drained", "amount_equals_old_orig",
]

FEATURES = PRE_TXN_FEATURES + POST_TXN_FEATURES


def signed_log1p(x):
    """log1p that keeps the sign, so negative balance errors stay meaningful."""
    return np.sign(x) * np.log1p(np.abs(x))


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add engineered columns to a PaySim-style dataframe (in place) and return it.

    Required columns: step, type, amount, oldbalanceOrg, newbalanceOrig,
    oldbalanceDest, newbalanceDest, nameDest.
    """
    # Balance-consistency errors (post-transaction)
    df["orig_balance_error"] = signed_log1p(df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"])
    df["dest_balance_error"] = signed_log1p(df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"])

    # Amount relative to balances (log1p tames extreme ratios)
    df["amount_to_orig_balance"] = np.log1p(df["amount"] / (df["oldbalanceOrg"] + 1.0))
    df["amount_to_dest_balance"] = np.log1p(df["amount"] / (df["oldbalanceDest"] + 1.0))

    # Behavioural flags
    drained = (df["newbalanceOrig"] == 0) & (df["oldbalanceOrg"] > 0)
    same_as_balance = np.isclose(df["amount"], df["oldbalanceOrg"]) & (df["oldbalanceOrg"] > 0)
    df["orig_drained"] = drained.astype(np.int8)
    df["amount_equals_old_orig"] = same_as_balance.astype(np.int8)
    df["dest_is_merchant"] = df["nameDest"].astype(str).str.startswith("M").astype(np.int8)
    df["hour_of_day"] = (df["step"] % 24).astype(np.int8)

    # Fixed one-hot encoding (works for a single row too)
    for t in TYPES:
        df[f"type_{t}"] = (df["type"] == t).astype(np.int8)
    return df


def nn_inputs(df: pd.DataFrame) -> np.ndarray:
    """Feature matrix for neural networks: money columns are signed-log transformed."""
    X = df[FEATURES].to_numpy(dtype=np.float64, copy=True)
    for col in MONEY_COLS:
        j = FEATURES.index(col)
        X[:, j] = signed_log1p(X[:, j])
    return X


def tree_explanation(explainer, X: pd.DataFrame):
    """SHAP Explanation for the fraud class, robust to different shap versions."""
    import shap
    sv = explainer(X, check_additivity=False)
    values, base = sv.values, sv.base_values
    if values.ndim == 3:
        values, base = values[:, :, 1], base[:, 1]
    return shap.Explanation(values=values, base_values=base,
                            data=np.asarray(X), feature_names=list(X.columns))


In [ ]:
from fraud_features import (
    FEATURES, PRE_TXN_FEATURES, POST_TXN_FEATURES, TYPE_COLS,
    engineer_features, nn_inputs,
)

df = engineer_features(df)

assert "isFraud" not in FEATURES and "isFlaggedFraud" not in FEATURES
assert np.isfinite(df[FEATURES].to_numpy(dtype=np.float64)).all()

print(f"Pre-transaction features:  {len(PRE_TXN_FEATURES)}")
print(f"Post-transaction features: {len(POST_TXN_FEATURES)}")
print(f"Total features:            {len(FEATURES)}")
df[FEATURES + ["isFraud"]].head()


### 5.1 Feature Check

Compare average feature values for legitimate and fraudulent transactions to confirm the features behave as intended.

In [ ]:
feature_means = df.groupby("isFraud")[FEATURES].mean().T
feature_means.columns = ["Legitimate", "Fraudulent"]
display(feature_means.round(3))

corr = df[FEATURES + ["isFraud"]].corr()["isFraud"].drop("isFraud").sort_values(key=abs, ascending=False)
corr.plot(kind="barh", figsize=(7, 6), title="Pearson correlation with isFraud").invert_yaxis()
plt.tight_layout()
plt.show()


---

## 6. Graph Construction

- **Nodes:** unique accounts
- **Edges:** transactions, directed origin → destination
- **Edge attributes / labels:** transaction features and `isFraud`

This section builds the global account mapping and measures how connected the graph is. Training uses time-window subgraphs built from this mapping in Section 8.

In [ ]:
# Map every account to an integer node ID
codes, accounts = pd.factorize(
    pd.concat([df["nameOrig"], df["nameDest"]], ignore_index=True)
)
n_tx = len(df)
df["src"] = codes[:n_tx].astype(np.int64)
df["dst"] = codes[n_tx:].astype(np.int64)
num_nodes = len(accounts)
del codes, accounts

print(f"Nodes (accounts):     {num_nodes:,}")
print(f"Edges (transactions): {n_tx:,}")


### 6.1 Graph Statistics

The earlier "average transactions per node" (edges ÷ nodes) is replaced by true degree statistics and connected components.

In [ ]:
out_deg = np.bincount(df["src"], minlength=num_nodes)
in_deg = np.bincount(df["dst"], minlength=num_nodes)
degree = out_deg + in_deg

print(f"Average degree:            {degree.mean():.2f}")
print(f"Nodes with degree 1:       {(degree == 1).mean() * 100:.1f}%")
print(f"Max in-degree:             {in_deg.max():,}")
print(f"Max out-degree:            {out_deg.max():,}")

# Connected components, ignoring edge direction
adjacency = coo_matrix(
    (np.ones(n_tx, dtype=np.int8), (df["src"].to_numpy(), df["dst"].to_numpy())),
    shape=(num_nodes, num_nodes),
)
n_components, component_of_node = connected_components(adjacency, directed=False)
component_sizes = np.bincount(component_of_node)
edge_component_size = component_sizes[component_of_node[df["src"].to_numpy()]]

lone_pair = edge_component_size == 2
print(f"\nConnected components:                  {n_components:,}")
print(f"Largest component (nodes):             {component_sizes.max():,}")
print(f"Edges that are an isolated A->B pair:  {lone_pair.mean() * 100:.1f}%")
print(f"  among fraud edges:                   {lone_pair[df['isFraud'].to_numpy() == 1].mean() * 100:.1f}%")

del adjacency, component_of_node, component_sizes, edge_component_size, lone_pair, out_deg, in_deg, degree


**Interpretation:** the share of isolated pairs tells you how much neighbourhood information a GNN can possibly use. The more edges that sit in tiny components, the less message passing can add beyond the transaction's own features.

### 6.2 Example Fraud Neighbourhoods

A few fraudulent transactions with every other transaction that touches the same two accounts.

In [ ]:
import networkx as nx

fraud_examples = df[df["isFraud"] == 1].sample(6, random_state=RANDOM_STATE)
touch = np.zeros(len(df), dtype=bool)
account_ids = np.unique(np.concatenate([fraud_examples["src"], fraud_examples["dst"]]))
touch |= df["src"].isin(account_ids).to_numpy() | df["dst"].isin(account_ids).to_numpy()
local = df.loc[touch, ["nameOrig", "nameDest", "isFraud"]]

G = nx.DiGraph()
for row in local.itertuples(index=False):
    G.add_edge(row.nameOrig, row.nameDest, fraud=row.isFraud)

edge_colors = ["crimson" if G[u][v]["fraud"] else "lightgray" for u, v in G.edges]
plt.figure(figsize=(9, 5))
pos = nx.spring_layout(G, seed=RANDOM_STATE, k=0.6)
nx.draw_networkx(G, pos, node_size=60, with_labels=False, edge_color=edge_colors,
                 arrows=True, arrowsize=8, width=1.4)
plt.title(f"Neighbourhoods of 6 fraudulent transactions ({G.number_of_edges()} edges; fraud in red)")
plt.axis("off")
plt.tight_layout()
plt.show()
del touch, local, G


---

## 7. Chronological Train / Validation / Test Split

Time-ordered splits mimic deployment: train on the past, tune on a later period, test on the latest period.

| Split | Role | Prevalence |
| --- | --- | --- |
| Train | fits models and the scaler | legitimate rows down-sampled, all fraud kept |
| Validation | early stopping, model choice, threshold | **natural** prevalence |
| Test | final one-time evaluation | **natural** prevalence |

Validation keeps the natural fraud rate so the decision threshold chosen on it is meaningful on the test period.

In [ ]:
test_start_step = int(df["step"].quantile(TEST_QUANTILE))
pre_test = df[df["step"] <= test_start_step]
val_start_step = int(pre_test["step"].quantile(VAL_QUANTILE))

train_full = df[df["step"] <= val_start_step]
val_raw = df[(df["step"] > val_start_step) & (df["step"] <= test_start_step)]
test_raw = df[df["step"] > test_start_step]

assert train_full["step"].max() < val_raw["step"].min()
assert val_raw["step"].max() < test_raw["step"].min()

def split_summary(name, d):
    return {"split": name, "steps": f"{int(d['step'].min())}-{int(d['step'].max())}",
            "transactions": len(d), "fraud": int(d["isFraud"].sum()),
            "fraud_%": round(d["isFraud"].mean() * 100, 3)}

display(pd.DataFrame([split_summary("train (full)", train_full),
                      split_summary("validation", val_raw),
                      split_summary("test", test_raw)]))


### 7.1 Down-sample Training Legitimate Transactions

All training fraud is kept. A fraction of legitimate training rows is sampled to keep training fast. Validation and test are **not** down-sampled. Class imbalance in the loss is handled in Section 9.

In [ ]:
train_fraud = train_full[train_full["isFraud"] == 1]
train_legit = train_full[train_full["isFraud"] == 0].sample(
    frac=TRAIN_LEGIT_FRAC, random_state=RANDOM_STATE
)
train_sample = (
    pd.concat([train_fraud, train_legit])
    .sort_values("step", kind="stable")
    .reset_index(drop=True)
)
val_df = val_raw.reset_index(drop=True)
test_df = test_raw.reset_index(drop=True)

print(f"Training sample: {len(train_sample):,} rows, {int(train_sample['isFraud'].sum()):,} fraud "
      f"({train_sample['isFraud'].mean() * 100:.2f}%)")
print(f"Validation:      {len(val_df):,} rows, {int(val_df['isFraud'].sum()):,} fraud "
      f"({val_df['isFraud'].mean() * 100:.3f}%)")
print(f"Test:            {len(test_df):,} rows, {int(test_df['isFraud'].sum()):,} fraud "
      f"({test_df['isFraud'].mean() * 100:.3f}%)")
del train_full, train_fraud, train_legit, pre_test, val_raw, test_raw

# The full dataframe is no longer needed; free memory (re-run Section 3 if you want it back)
del df
gc.collect()


### 7.2 Feature Scaling for the Neural Networks

`StandardScaler` is fitted on the training sample only (after the signed-log transform of money columns) and applied unchanged to validation and test. Tree models use raw features and need no scaling.

In [ ]:
scaler = StandardScaler().fit(nn_inputs(train_sample))

def scaled_inputs(d: pd.DataFrame) -> np.ndarray:
    """Scaled, clipped float32 features for the neural networks."""
    X = scaler.transform(nn_inputs(d))
    return np.clip(X, -10, 10).astype(np.float32)

X_check = scaled_inputs(train_sample)
print("Scaled training matrix:", X_check.shape, X_check.dtype)
print("Max |value| after clipping:", float(np.abs(X_check).max()))
del X_check


---

## 8. Graph Windows and Model Definitions

### 8.1 Time-Window Graphs

One graph is built per 24-hour window of transactions, separately for train, validation and test.

- Message passing runs over **reverse edges as well**, so both the origin and the destination account receive information.
- **Node features** = average of the scaled features of the account's transactions in that window, plus log in-degree and out-degree. These use transaction features only, never labels.
- **Edge features** = scaled transaction features. They enter the GAT attention and the final edge classifier.
- Validation and test graphs contain only their own period's transactions, so evaluation is **inductive** and never sees training labels or later data.

In [ ]:
def make_windows(part_df: pd.DataFrame, window_steps: int = WINDOW_STEPS):
    """Split a time-sorted dataframe into consecutive windows of `window_steps` hours."""
    start = int(part_df["step"].min())
    window_id = (part_df["step"] - start) // window_steps
    return [g.reset_index(drop=True) for _, g in part_df.groupby(window_id, sort=True)]


def build_graph(w_df: pd.DataFrame) -> Data:
    """Build one account graph (edge classification) from a window of transactions."""
    n_edges = len(w_df)
    endpoints = np.concatenate([w_df["src"].to_numpy(), w_df["dst"].to_numpy()])
    nodes, inverse = np.unique(endpoints, return_inverse=True)
    inverse = inverse.reshape(-1)
    n_nodes = len(nodes)

    src = torch.from_numpy(inverse[:n_edges].astype(np.int64))
    dst = torch.from_numpy(inverse[n_edges:].astype(np.int64))
    edge_attr = torch.from_numpy(scaled_inputs(w_df))

    # Account features: mean of incident transaction features + degree
    node_sum = torch.zeros(n_nodes, edge_attr.shape[1])
    node_count = torch.zeros(n_nodes)
    ones = torch.ones(n_edges)
    for idx in (src, dst):
        node_sum.index_add_(0, idx, edge_attr)
        node_count.index_add_(0, idx, ones)
    node_mean = node_sum / node_count.clamp(min=1).unsqueeze(1)
    out_degree = torch.bincount(src, minlength=n_nodes).float()
    in_degree = torch.bincount(dst, minlength=n_nodes).float()
    x = torch.cat([node_mean, torch.log1p(out_degree)[:, None], torch.log1p(in_degree)[:, None]], dim=1)

    edge_index = torch.stack([src, dst])
    return Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=torch.from_numpy(w_df["isFraud"].to_numpy(dtype=np.float32)),
        src=src,
        dst=dst,
        # Message passing uses both directions so origin accounts also receive messages
        mp_edge_index=torch.cat([edge_index, edge_index.flip(0)], dim=1),
        mp_edge_attr=torch.cat([edge_attr, edge_attr], dim=0),
    )


train_windows = make_windows(train_sample)
val_windows = make_windows(val_df)
test_windows = make_windows(test_df)

# Canonical row order for every model's predictions
train_df = pd.concat(train_windows, ignore_index=True)
val_df = pd.concat(val_windows, ignore_index=True)
test_df = pd.concat(test_windows, ignore_index=True)

train_graphs = [build_graph(w) for w in train_windows]
val_graphs = [build_graph(w) for w in val_windows]
test_graphs = [build_graph(w) for w in test_windows]

graph_summary = pd.DataFrame([
    {"split": name, "graphs": len(gs),
     "nodes": sum(g.num_nodes for g in gs),
     "edges": sum(g.edge_index.shape[1] for g in gs),
     "fraud edges": int(sum(g.y.sum().item() for g in gs))}
    for name, gs in [("train", train_graphs), ("validation", val_graphs), ("test", test_graphs)]
])
display(graph_summary)

NODE_DIM = train_graphs[0].x.shape[1]
EDGE_DIM = train_graphs[0].edge_attr.shape[1]
print(f"Node feature dimension: {NODE_DIM}   Edge feature dimension: {EDGE_DIM}")


### 8.2 GCN and GAT Edge Classifiers

Both models share the same structure:

1. Two graph-convolution layers turn each account's features into an **account embedding** that depends on its neighbours.
2. An MLP **edge classifier** receives `[source embedding, destination embedding, transaction features]` and outputs a fraud logit.

| Model | Convolution | Uses transaction features inside message passing? |
| --- | --- | --- |
| **GCN** | `GCNConv`: fixed, degree-normalised neighbour averaging | No |
| **GAT** | `GATConv`: learned attention weights over neighbours | Yes (`edge_dim`) |

In [ ]:
class EdgeGNN(nn.Module):
    """GCN or GAT encoder with an MLP head that classifies transactions (edges)."""

    def __init__(self, kind, node_dim, edge_dim, hidden=HIDDEN_DIM, heads=GAT_HEADS, dropout=DROPOUT):
        super().__init__()
        self.kind = kind
        self.drop = nn.Dropout(dropout)
        if kind == "gcn":
            self.conv1 = GCNConv(node_dim, hidden)
            self.conv2 = GCNConv(hidden, hidden)
        elif kind == "gat":
            self.conv1 = GATConv(node_dim, hidden // heads, heads=heads, dropout=dropout, edge_dim=edge_dim)
            self.conv2 = GATConv(hidden, hidden, heads=1, concat=False, dropout=dropout, edge_dim=edge_dim)
        else:
            raise ValueError("kind must be 'gcn' or 'gat'")
        self.head = nn.Sequential(
            nn.Linear(2 * hidden + edge_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def _conv(self, conv, x, g):
        if self.kind == "gat":
            return conv(x, g.mp_edge_index, edge_attr=g.mp_edge_attr)
        return conv(x, g.mp_edge_index)

    def encode(self, g):
        """Account embeddings."""
        h = F.elu(self._conv(self.conv1, g.x, g))
        h = self.drop(h)
        return F.elu(self._conv(self.conv2, h, g))

    def classify(self, h, src, dst, edge_attr):
        """Fraud logit per transaction from account embeddings + transaction features."""
        return self.head(torch.cat([h[src], h[dst], edge_attr], dim=1)).squeeze(-1)

    def forward(self, g):
        return self.classify(self.encode(g), g.src, g.dst, g.edge_attr)


for kind in ("gcn", "gat"):
    m = EdgeGNN(kind, NODE_DIM, EDGE_DIM)
    print(f"{kind.upper()}: {sum(p.numel() for p in m.parameters()):,} trainable parameters")
del m


---

## 9. Training

### 9.1 Evaluation Helpers

Helper functions used for every model, so all models are scored identically:

- `evaluate` reports ROC-AUC, PR-AUC, precision, recall, F1 and top-k alert quality.
- `best_f1_threshold` picks a decision threshold on the **validation** set only.
- Top-k metrics ask: "if analysts review only the top *k*% highest-risk transactions, how many frauds do they catch?"

In [ ]:
def best_f1_threshold(y, p):
    precision, recall, thresholds = precision_recall_curve(y, p)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    return float(thresholds[int(np.argmax(f1))])


def top_fraction_stats(y, p, fraction):
    """Recall and precision when only the top `fraction` highest-scored rows are flagged."""
    k = max(int(round(len(y) * fraction)), 1)
    top = np.argsort(-p, kind="stable")[:k]
    hits = y[top].sum()
    return hits / max(y.sum(), 1), hits / k


def evaluate(y, p, threshold, top_fraction=0.005):
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    recall_k, precision_k = top_fraction_stats(y, p, top_fraction)
    return {
        "ROC-AUC": roc_auc_score(y, p),
        "PR-AUC": average_precision_score(y, p),
        "Precision": precision, "Recall": recall, "F1": f1,
        f"Recall@top{top_fraction:.1%}": recall_k,
        f"Precision@top{top_fraction:.1%}": precision_k,
        "TP": int(tp), "FP": int(fp), "FN": int(fn),
    }


# score store: name -> {"val": (y, p), "test": (y, p)}
scores = {}

def register_model(name, p_val, p_test):
    scores[name] = {"val": (val_df["isFraud"].to_numpy(), np.asarray(p_val)),
                    "test": (test_df["isFraud"].to_numpy(), np.asarray(p_test))}


### 9.2 Baselines: Random Forest

Two transaction-level baselines are trained on the same training sample:

- **RF (all features)**: uses every feature, including post-transaction balances.
- **RF (pre-transaction only)**: uses only features known when a transaction is submitted. The gap between the two shows how much performance depends on post-transaction information.

`class_weight="balanced_subsample"` handles the imbalance.

In [ ]:
def fit_random_forest(features):
    rf = RandomForestClassifier(
        n_estimators=150, max_depth=16, min_samples_leaf=2,
        class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf.fit(train_df[features], train_df["isFraud"])
    return rf

start = time.time()
rf_all = fit_random_forest(FEATURES)
rf_pre = fit_random_forest(PRE_TXN_FEATURES)
print(f"Random Forests trained in {time.time() - start:.0f}s")

register_model(
    "RF (all features)",
    rf_all.predict_proba(val_df[FEATURES])[:, 1],
    rf_all.predict_proba(test_df[FEATURES])[:, 1],
)
register_model(
    "RF (pre-transaction only)",
    rf_pre.predict_proba(val_df[PRE_TXN_FEATURES])[:, 1],
    rf_pre.predict_proba(test_df[PRE_TXN_FEATURES])[:, 1],
)

for name in ("RF (all features)", "RF (pre-transaction only)"):
    y_v, p_v = scores[name]["val"]
    print(f"{name:28s} validation PR-AUC = {average_precision_score(y_v, p_v):.4f}")


### 9.3 Train GCN and GAT

- **Loss:** `BCEWithLogitsLoss` with `pos_weight = sqrt(negatives / positives)` computed from the training graphs. A square root is used because the training sample was already enriched with fraud; the exact operating point is set later by the validation threshold.
- **Optimiser:** Adam with weight decay.
- **Each epoch:** one gradient step per training window graph, then validation PR-AUC on the natural-prevalence validation graphs.
- **Early stopping:** the best validation PR-AUC checkpoint is restored.

On the full PaySim data, expect several minutes per model on a CPU (each epoch scores all validation windows).

In [ ]:
def predict_graphs(model, graphs):
    """Fraud probabilities and labels for a list of graphs (rows follow window order)."""
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for g in graphs:
            g = g.to(DEVICE)
            ps.append(torch.sigmoid(model(g)).cpu().numpy())
            ys.append(g.y.cpu().numpy())
    return np.concatenate(ys).astype(int), np.concatenate(ps)


train_labels = np.concatenate([g.y.numpy() for g in train_graphs])
n_pos, n_neg = int(train_labels.sum()), int((train_labels == 0).sum())
POS_WEIGHT = float(np.sqrt(n_neg / max(n_pos, 1)))
print(f"Training edges: {len(train_labels):,}  fraud: {n_pos:,}  legitimate: {n_neg:,}")
print(f"pos_weight = sqrt({n_neg:,} / {n_pos:,}) = {POS_WEIGHT:.2f}")


def train_gnn(kind):
    torch.manual_seed(RANDOM_STATE)
    model = EdgeGNN(kind, NODE_DIM, EDGE_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))

    history = {"loss": [], "val_pr_auc": []}
    best_pr, best_state, bad_epochs = -1.0, None, 0
    order = list(range(len(train_graphs)))

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        random.shuffle(order)
        total_loss, total_edges = 0.0, 0
        for i in order:
            g = train_graphs[i].to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(g), g.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * g.y.numel()
            total_edges += g.y.numel()

        y_val, p_val = predict_graphs(model, val_graphs)
        val_pr = average_precision_score(y_val, p_val)
        history["loss"].append(total_loss / total_edges)
        history["val_pr_auc"].append(val_pr)
        print(f"[{kind.upper()}] epoch {epoch:02d}  train loss {history['loss'][-1]:.4f}  val PR-AUC {val_pr:.4f}")

        if val_pr > best_pr + 1e-4:
            best_pr, best_state, bad_epochs = val_pr, copy.deepcopy(model.state_dict()), 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"[{kind.upper()}] early stopping (best val PR-AUC {best_pr:.4f})")
                break

    model.load_state_dict(best_state)
    return model, history


gnn_models, gnn_history = {}, {}
for kind in ("gcn", "gat"):
    start = time.time()
    gnn_models[kind.upper()], gnn_history[kind.upper()] = train_gnn(kind)
    print(f"{kind.upper()} trained in {time.time() - start:.0f}s\n")

for name, model in gnn_models.items():
    _, p_val = predict_graphs(model, val_graphs)
    _, p_test = predict_graphs(model, test_graphs)
    register_model(name, p_val, p_test)


### 9.4 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, history in gnn_history.items():
    axes[0].plot(history["loss"], label=name)
    axes[1].plot(history["val_pr_auc"], label=name)
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Weighted BCE loss")
axes[1].set(title="Validation PR-AUC", xlabel="Epoch", ylabel="PR-AUC")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()


---

## 10. Model Evaluation and Sanity Checks

Every model uses the same procedure:

1. Choose its decision threshold on the **validation** set (maximum F1).
2. Apply that threshold **unchanged** to the test set.

The test set is used once, for the final report.

### 10.1 Validation and Test Results

In [ ]:
thresholds = {}
rows = []
for name, split_scores in scores.items():
    y_val, p_val = split_scores["val"]
    thresholds[name] = best_f1_threshold(y_val, p_val)
    for split in ("val", "test"):
        y, p = split_scores[split]
        rows.append({"Model": name, "Split": "validation" if split == "val" else "test",
                     "Threshold": thresholds[name], **evaluate(y, p, thresholds[name])})

results = pd.DataFrame(rows)
fmt = {c: "{:.4f}" for c in ["Threshold", "ROC-AUC", "PR-AUC", "Precision", "Recall", "F1"]}
fmt.update({c: "{:.3f}" for c in results.columns if "@top" in c})

print("Validation (threshold tuned here)")
display(results[results["Split"] == "validation"].drop(columns="Split").style.format(fmt).hide(axis="index"))
print("Test (final, threshold fixed from validation)")
display(results[results["Split"] == "test"].drop(columns="Split").style.format(fmt).hide(axis="index"))


### 10.2 Precision–Recall Curves (Test)

In [ ]:
plt.figure(figsize=(8, 5))
for name, split_scores in scores.items():
    y, p = split_scores["test"]
    precision, recall, _ = precision_recall_curve(y, p)
    plt.plot(recall, precision, label=f"{name} (AP={average_precision_score(y, p):.3f})")
plt.axhline(test_df["isFraud"].mean(), color="gray", linestyle="--", label="No-skill (prevalence)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves on the Chronological Test Set")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 10.3 Confusion Matrices (Test)

False positives are legitimate transactions sent to analysts; false negatives are missed fraud.

In [ ]:
fig, axes = plt.subplots(1, len(scores), figsize=(4.2 * len(scores), 4))
for ax, (name, split_scores) in zip(np.atleast_1d(axes), scores.items()):
    y, p = split_scores["test"]
    cm = confusion_matrix(y, (p >= thresholds[name]).astype(int), labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["Legit", "Fraud"]).plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(name, fontsize=10)
plt.tight_layout()
plt.show()


### 10.4 Alert Quality: Top-k Review

Analysts can only review a limited number of alerts. This table shows, for each review budget, what share of all fraud is caught and how many flagged transactions are truly fraud.

In [ ]:
topk_rows = []
for name, split_scores in scores.items():
    y, p = split_scores["test"]
    for fraction in (0.001, 0.005, 0.01):
        recall_k, precision_k = top_fraction_stats(y, p, fraction)
        topk_rows.append({"Model": name, "Review budget": f"top {fraction:.1%}",
                          "Alerts": int(round(len(y) * fraction)),
                          "Fraud recall": recall_k, "Alert precision": precision_k})

topk_table = pd.DataFrame(topk_rows)
display(topk_table.pivot(index="Model", columns="Review budget", values=["Fraud recall", "Alert precision"]).round(3))


### 10.5 Sanity Checks

PaySim is synthetic and fraud follows strong simulator patterns, so very high scores must be checked before they are believed.

1. **Post-transaction dependence:** compare RF (all features) with RF (pre-transaction only).
2. **Train/test duplicates:** do identical feature rows appear in both?
3. **Per-type performance:** is the model equally good on TRANSFER and CASH\_OUT?
4. **Feature importance:** what is the forest actually relying on?

In [ ]:
# 1. Dependence on post-transaction features
for split in ("val", "test"):
    y, p_all = scores["RF (all features)"][split]
    _, p_pre = scores["RF (pre-transaction only)"][split]
    print(f"{split:5s}  PR-AUC all features: {average_precision_score(y, p_all):.4f}   "
          f"pre-transaction only: {average_precision_score(y, p_pre):.4f}")

# 2. Identical feature rows across train and test
def row_hashes(d):
    return pd.util.hash_pandas_object(d[FEATURES].round(2), index=False).to_numpy()

overlap = np.isin(row_hashes(test_df), row_hashes(train_df))
is_fraud_test = test_df["isFraud"].to_numpy() == 1
print(f"\nTest rows whose feature vector also appears in training: {overlap.mean() * 100:.2f}%")
print(f"  among test fraud rows:                                 {overlap[is_fraud_test].mean() * 100:.2f}%")

# 3. Per-type recall at each model's threshold
per_type = []
for name in scores:
    y, p = scores[name]["test"]
    pred = p >= thresholds[name]
    for t in ("TRANSFER", "CASH_OUT"):
        mask = (test_df["type"] == t).to_numpy() & (y == 1)
        per_type.append({"Model": name, "Type": t, "Fraud cases": int(mask.sum()),
                         "Recall": pred[mask].mean() if mask.any() else np.nan})
display(pd.DataFrame(per_type).pivot(index="Model", columns="Type", values="Recall").round(4))

# 4. Random Forest feature importance
importance = pd.Series(rf_all.feature_importances_, index=FEATURES).sort_values()
importance.plot(kind="barh", figsize=(7, 6), title="Random Forest (all features): impurity importance")
plt.tight_layout()
plt.show()


### 10.6 Summary of Findings

The cell below generates the key comparisons from the computed results, so the text always matches the numbers.

In [ ]:
val_pr = {n: average_precision_score(*s["val"]) for n, s in scores.items()}
test_pr = {n: average_precision_score(*s["test"]) for n, s in scores.items()}
champion = max(val_pr, key=val_pr.get)
best_gnn_name = max(("GCN", "GAT"), key=val_pr.get)
best_graph_vs_rf = test_pr[best_gnn_name] - test_pr["RF (all features)"]

print(f"Champion by validation PR-AUC: {champion}")
print(f"Best graph model:              {best_gnn_name} (test PR-AUC {test_pr[best_gnn_name]:.4f})")
print(f"RF (all features) test PR-AUC: {test_pr['RF (all features)']:.4f}")
print(f"Graph model minus RF:          {best_graph_vs_rf:+.4f} PR-AUC")
print(f"RF all vs pre-transaction:     {test_pr['RF (all features)'] - test_pr['RF (pre-transaction only)']:+.4f} PR-AUC")

if best_graph_vs_rf > 0.005:
    print("\n-> The graph model adds measurable value over the transaction-level baseline.")
elif best_graph_vs_rf < -0.005:
    print("\n-> The transaction-level baseline outperforms the graph models on this data.")
else:
    print("\n-> Graph and baseline models perform about the same on this data.")


**How to read these results (for your write-up)**

- PaySim fraud follows a simulator pattern (large TRANSFER/CASH\_OUT that drains the origin account), so tree models on balance features can score very high. Report this honestly rather than as real-world performance.
- If the graph models do not beat the baseline, a likely cause is the graph structure measured in Section 6.1: if most accounts appear only once, there is little neighbourhood information to learn from. This is a dataset limitation, not an implementation error.
- The pre-transaction RF shows what is achievable without post-transaction balances.
- The graph adds the most value when fraud forms connected patterns (rings, mule accounts, repeated destinations). Real transaction networks have far richer structure than PaySim.

---

## 11. SHAP Explainability

SHAP assigns each input a contribution to a single prediction.

- **11.1 to 11.3 Random Forest:** `TreeExplainer` gives exact SHAP values for every transaction feature (global view, individual transactions, pre- vs post-transaction share).
- **11.4 Graph model (GCN/GAT):** a graph model's prediction depends on account embeddings as well as the transaction. SHAP is applied to the model's classifier head with the transaction features as individual inputs and the **source-account embedding** and **destination-account embedding** as two grouped inputs. This shows how much of a prediction comes from the transaction row itself versus account-level context.

### 11.1 Random Forest: Global and Local Explanations

In [ ]:
import shap
from fraud_features import tree_explanation

rng = np.random.default_rng(RANDOM_STATE)
test_fraud_idx = np.flatnonzero(test_df["isFraud"].to_numpy() == 1)
test_legit_idx = np.flatnonzero(test_df["isFraud"].to_numpy() == 0)
shap_idx = np.concatenate([
    rng.choice(test_fraud_idx, min(300, len(test_fraud_idx)), replace=False),
    rng.choice(test_legit_idx, 300, replace=False),
])

X_shap = test_df.loc[shap_idx, FEATURES]
rf_explainer = shap.TreeExplainer(rf_all)
rf_explanation = tree_explanation(rf_explainer, X_shap)

shap.plots.beeswarm(rf_explanation, max_display=12, show=False)
plt.title("SHAP summary: Random Forest (fraud class)")
plt.tight_layout()
plt.show()

shap.plots.bar(rf_explanation, max_display=12, show=False)
plt.title("Mean |SHAP| per feature")
plt.tight_layout()
plt.show()


### 11.2 Random Forest: Individual Transactions

One correctly detected fraud and one false alarm, explained.

In [ ]:
rf_test_scores = scores["RF (all features)"]["test"][1]
y_test_arr = test_df["isFraud"].to_numpy()

true_positive_pos = int(np.argmax(np.where(y_test_arr == 1, rf_test_scores, -1)))
false_positive_pos = int(np.argmax(np.where(y_test_arr == 0, rf_test_scores, -1)))

for title, pos in [("Detected fraud (highest-scoring true fraud)", true_positive_pos),
                   ("False alarm (highest-scoring legitimate transaction)", false_positive_pos)]:
    explanation = tree_explanation(rf_explainer, test_df.loc[[pos], FEATURES])[0]
    print(f"{title}: type={test_df.loc[pos, 'type']}, amount={test_df.loc[pos, 'amount']:,.0f}, "
          f"score={rf_test_scores[pos]:.3f}")
    shap.plots.waterfall(explanation, max_display=10, show=False)
    plt.tight_layout()
    plt.show()


### 11.3 Pre-transaction vs Post-transaction Features

Share of total SHAP importance coming from features known at submission time versus after the transaction completes.

In [ ]:
mean_abs_shap = pd.Series(np.abs(rf_explanation.values).mean(axis=0), index=FEATURES)
group_share = pd.Series({
    "Pre-transaction features": mean_abs_shap[PRE_TXN_FEATURES].sum(),
    "Post-transaction features": mean_abs_shap[POST_TXN_FEATURES].sum(),
})
display((group_share / group_share.sum() * 100).round(1).to_frame("Share of mean |SHAP| (%)"))


### 11.4 Graph Model: Transaction vs Network-Context Attribution

Using the best graph model by validation PR-AUC. The background is a set of typical legitimate transactions, so SHAP values explain why a transaction scores higher than a typical legitimate one.

In [ ]:
best_gnn = gnn_models[best_gnn_name].to("cpu").eval()
n_feat = len(FEATURES)

# Test window with the most fraud
window_idx = int(np.argmax([int(g.y.sum().item()) for g in test_graphs]))
g = test_graphs[window_idx]
with torch.no_grad():
    embeddings = best_gnn.encode(g)

y_window = g.y.numpy()
fraud_pos = np.flatnonzero(y_window == 1)
legit_pos = np.flatnonzero(y_window == 0)
explain_pos = np.concatenate([
    rng.choice(fraud_pos, min(30, len(fraud_pos)), replace=False),
    rng.choice(legit_pos, 30, replace=False),
])
background_pos = rng.choice(legit_pos, 40, replace=False)
pool_pos = np.concatenate([background_pos, explain_pos])

# Pools: the explainer swaps columns between rows, so embeddings are looked up by index
src_pool = embeddings[g.src[pool_pos]].numpy()
dst_pool = embeddings[g.dst[pool_pos]].numpy()
feat_pool = g.edge_attr[pool_pos].numpy()

def make_rows(pool_rows):
    idx = pool_rows.astype(float)
    return np.column_stack([feat_pool[pool_rows], idx, idx])   # [features | src ctx index | dst ctx index]

rows_background = make_rows(np.arange(len(background_pos)))
rows_explain = make_rows(np.arange(len(background_pos), len(pool_pos)))

def gnn_head_predict(Z):
    Z = np.asarray(Z)
    s = np.rint(Z[:, n_feat]).astype(int)
    d = np.rint(Z[:, n_feat + 1]).astype(int)
    inputs = np.concatenate([src_pool[s], dst_pool[d], Z[:, :n_feat]], axis=1)
    with torch.no_grad():
        logits = best_gnn.head(torch.tensor(inputs, dtype=torch.float32)).squeeze(-1)
    return torch.sigmoid(logits).numpy()

gnn_explainer = shap.KernelExplainer(gnn_head_predict, rows_background)
gnn_shap = gnn_explainer.shap_values(rows_explain, nsamples=300, silent=True)
gnn_feature_names = FEATURES + ["SOURCE account context", "DESTINATION account context"]

shap.summary_plot(gnn_shap, rows_explain, feature_names=gnn_feature_names,
                  plot_type="bar", max_display=12, show=False)
plt.title(f"{best_gnn_name}: mean |SHAP| (transaction features vs account context)")
plt.tight_layout()
plt.show()

mean_abs = np.abs(gnn_shap).mean(axis=0)
attribution = pd.Series({
    "Transaction features": mean_abs[:n_feat].sum(),
    "Source-account context": mean_abs[n_feat],
    "Destination-account context": mean_abs[n_feat + 1],
})
display((attribution / attribution.sum() * 100).round(1).to_frame("Share of mean |SHAP| (%)"))

best_gnn.to(DEVICE)


**Reading the graph attribution:** an account embedding is computed from the account's transactions in the window, *including the transaction being scored*. A large "account context" share therefore shows reliance on account-level behaviour, but it does not by itself prove that *neighbouring* accounts matter. Whether the network adds value is answered by comparing the GNNs with the Random Forest in Section 10.

---

## 12. Save Artifacts and Streamlit Dashboard

### 12.1 Save Models, Scores and Dashboard Data

The dashboard needs:

- the Random Forest bundle (for live scoring and SHAP),
- a comparison table of all models,
- a scored sample of the **test period** (all fraud + the highest-scoring transactions + a random sample) with GCN/GAT scores precomputed.

Live scoring in the app uses the Random Forest because GNN scoring needs the surrounding transaction graph; GNN scores are shown for the saved test-period sample.

In [ ]:
SCORE_COLUMNS = {"Random Forest": "score_rf", "GCN": "score_gcn", "GAT": "score_gat"}
score_threshold = {
    "score_rf": thresholds["RF (all features)"],
    "score_gcn": thresholds["GCN"],
    "score_gat": thresholds["GAT"],
}

# ---- Models ----
joblib.dump(
    {"model": rf_all, "features": FEATURES, "threshold": thresholds["RF (all features)"],
     "thresholds": score_threshold, "score_columns": SCORE_COLUMNS},
    MODEL_DIR / "fraud_rf.joblib",
)
joblib.dump(scaler, MODEL_DIR / "nn_scaler.joblib")
for name, model in gnn_models.items():
    torch.save(
        {"state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
         "kind": name.lower(), "node_dim": NODE_DIM, "edge_dim": EDGE_DIM,
         "hidden": HIDDEN_DIM, "heads": GAT_HEADS, "threshold": thresholds[name],
         "features": FEATURES},
        MODEL_DIR / f"{name.lower()}_model.pt",
    )

# ---- Comparison table ----
results.to_csv(MODEL_DIR / "model_comparison.csv", index=False)

# ---- Scored test sample for the dashboard ----
test_scored = test_df.copy()
test_scored["score_rf"] = scores["RF (all features)"]["test"][1]
test_scored["score_gcn"] = scores["GCN"]["test"][1]
test_scored["score_gat"] = scores["GAT"]["test"][1]

max_score = test_scored[["score_rf", "score_gcn", "score_gat"]].max(axis=1).to_numpy()
fraud_pos = np.flatnonzero(test_scored["isFraud"].to_numpy() == 1)
top_pos = np.argsort(-max_score)[:20_000]
random_pos = rng.choice(len(test_scored), min(30_000, len(test_scored)), replace=False)
dash_pos = np.unique(np.concatenate([fraud_pos, top_pos, random_pos]))

base_cols = ["step", "type", "amount", "nameOrig", "nameDest", "oldbalanceOrg",
             "newbalanceOrig", "oldbalanceDest", "newbalanceDest", "isFraud"]
dash_cols = list(dict.fromkeys(base_cols + FEATURES + ["score_rf", "score_gcn", "score_gat"]))
dashboard_df = test_scored.iloc[dash_pos][dash_cols]
dashboard_df.to_csv(MODEL_DIR / "dashboard_scored.csv.gz", index=False)

print("Saved files:")
for path in sorted(MODEL_DIR.iterdir()):
    print(f"  {path.name:28s} {path.stat().st_size / 1e6:8.2f} MB")
print(f"\nDashboard sample: {len(dashboard_df):,} test transactions, {int(dashboard_df['isFraud'].sum()):,} fraud")


### 12.2 Verify the Saved Random Forest

Reload the bundle and confirm that predictions are identical, so the dashboard behaves exactly like the notebook.

In [ ]:
bundle = joblib.load(MODEL_DIR / "fraud_rf.joblib")
sample = val_df[bundle["features"]].iloc[:200]
difference = np.abs(rf_all.predict_proba(sample)[:, 1] - bundle["model"].predict_proba(sample)[:, 1]).max()
print("Features saved:       ", len(bundle["features"]))
print("Threshold saved:      ", round(bundle["threshold"], 4))
print("Max prediction diff:  ", difference)
assert difference == 0


### 12.3 Streamlit Dashboard

`app.py` provides three views:

| Tab | What it does |
| --- | --- |
| **Risk overview** | alert counts by risk tier, recall on the sample, score distribution, alerts by transaction type, model comparison |
| **Investigate alerts** | ranked list of suspicious transactions, SHAP explanation of the selected alert, account neighbourhood graph |
| **Score a transaction** | enter transaction details, get fraud probability, risk tier and a SHAP waterfall |

Risk tiers are defined relative to the model's validation-tuned threshold: **High** ≥ threshold, **Medium** ≥ half the threshold, **Low** below.

In [ ]:
%%writefile app.py
"""Streamlit dashboard: explainable fraud risk monitoring."""
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import shap
import streamlit as st

from fraud_features import FEATURES, TYPES, engineer_features, tree_explanation

MODEL_DIR = Path("saved_models")
st.set_page_config(page_title="Fraud Risk Dashboard", page_icon="🛡️", layout="wide")


@st.cache_resource
def load_model():
    bundle = joblib.load(MODEL_DIR / "fraud_rf.joblib")
    return bundle, shap.TreeExplainer(bundle["model"])


@st.cache_data
def load_scored():
    return pd.read_csv(MODEL_DIR / "dashboard_scored.csv.gz")


@st.cache_data
def load_comparison():
    return pd.read_csv(MODEL_DIR / "model_comparison.csv")


try:
    bundle, explainer = load_model()
    scored = load_scored()
    comparison = load_comparison()
except FileNotFoundError:
    st.error("Model files not found. Run the notebook (Section 12.1) first so `saved_models/` exists.")
    st.stop()

SCORE_COLUMNS = bundle["score_columns"]


def risk_tier(score, threshold):
    return np.where(score >= threshold, "High", np.where(score >= threshold / 2, "Medium", "Low"))


def shap_waterfall(row_features: pd.DataFrame):
    explanation = tree_explanation(explainer, row_features)[0]
    plt.figure()
    shap.plots.waterfall(explanation, max_display=12, show=False)
    st.pyplot(plt.gcf(), clear_figure=True)


def draw_neighbourhood(row_id, score_col):
    row = scored.loc[row_id]
    accounts = {row["nameOrig"], row["nameDest"]}
    related = scored[scored["nameOrig"].isin(accounts) | scored["nameDest"].isin(accounts)].head(40)
    graph = nx.DiGraph()
    for _, t in related.iterrows():
        graph.add_edge(t["nameOrig"], t["nameDest"], score=t[score_col])
    colors = ["crimson" if graph[u][v]["score"] >= threshold else "gray" for u, v in graph.edges]
    node_colors = ["gold" if n in accounts else "lightsteelblue" for n in graph.nodes]
    fig, ax = plt.subplots(figsize=(6, 4))
    pos = nx.spring_layout(graph, seed=0)
    nx.draw_networkx(graph, pos, ax=ax, node_size=140, with_labels=False,
                     edge_color=colors, node_color=node_colors, arrows=True, arrowsize=10)
    ax.set_title("Accounts around the selected transaction (red = above threshold)")
    ax.axis("off")
    st.pyplot(fig, clear_figure=True)
    st.caption(f"{len(related)} sampled transactions touch these accounts.")


# ------------------------------------------------------------------ sidebar
st.sidebar.title("🛡️ Fraud Risk Dashboard")
model_label = st.sidebar.selectbox("Score shown", list(SCORE_COLUMNS))
score_col = SCORE_COLUMNS[model_label]
threshold = bundle["thresholds"][score_col]
st.sidebar.metric("Decision threshold", f"{threshold:.3f}")
st.sidebar.caption("Threshold was tuned on the validation period (maximum F1). "
                   "Tiers: High >= threshold, Medium >= threshold / 2, otherwise Low.")

view = scored.copy()
view["score"] = view[score_col]
view["tier"] = risk_tier(view["score"], threshold)

tab_overview, tab_investigate, tab_score = st.tabs(
    ["Risk overview", "Investigate alerts", "Score a transaction"]
)

# ------------------------------------------------------------------ overview
with tab_overview:
    st.subheader(f"Test-period risk overview ({model_label})")
    st.caption("The sample contains all fraud plus the highest-scored and a random set of other "
               "transactions, so counts are not population rates. Recall is valid because all fraud is included.")

    n_fraud = int(view["isFraud"].sum())
    caught = int(((view["tier"] == "High") & (view["isFraud"] == 1)).sum())
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Transactions in sample", f"{len(view):,}")
    c2.metric("High-risk alerts", f"{int((view['tier'] == 'High').sum()):,}")
    c3.metric("Confirmed fraud in sample", f"{n_fraud:,}")
    c4.metric("Fraud caught as High", f"{caught / max(n_fraud, 1):.1%}")

    left, right = st.columns(2)
    with left:
        st.markdown("**Transactions by risk tier**")
        tiers = view["tier"].value_counts().reindex(["High", "Medium", "Low"]).fillna(0)
        st.bar_chart(tiers)
        st.markdown("**High-risk alerts by transaction type**")
        st.bar_chart(view[view["tier"] == "High"]["type"].value_counts())
    with right:
        st.markdown("**Score distribution by true label**")
        fig, ax = plt.subplots(figsize=(6, 3.4))
        ax.hist(view.loc[view["isFraud"] == 0, "score"], bins=40, alpha=0.6, label="Legitimate")
        ax.hist(view.loc[view["isFraud"] == 1, "score"], bins=40, alpha=0.6, label="Fraud")
        ax.axvline(threshold, color="black", linestyle="--", label="Threshold")
        ax.set_yscale("log")
        ax.set_xlabel("Fraud score")
        ax.legend()
        st.pyplot(fig, clear_figure=True)

    st.markdown("**Model comparison (validation threshold applied to both splits)**")
    st.dataframe(comparison.round(4), width="stretch", hide_index=True)

# ------------------------------------------------------------------ investigate
with tab_investigate:
    st.subheader("Investigate alerts")
    f1, f2 = st.columns(2)
    types = f1.multiselect("Transaction type", sorted(view["type"].unique()),
                           default=sorted(view["type"].unique()))
    min_score = f2.slider("Minimum score", 0.0, 1.0, float(min(threshold, 0.99)), 0.01)

    alerts = view[view["type"].isin(types) & (view["score"] >= min_score)].sort_values("score", ascending=False)
    st.write(f"{len(alerts):,} transactions match. Showing the top 200.")
    table_cols = ["step", "type", "amount", "nameOrig", "nameDest", "score", "tier", "isFraud"]
    st.dataframe(alerts[table_cols].head(200), width="stretch")

    if len(alerts):
        options = alerts.index[:200].tolist()
        chosen = st.selectbox(
            "Select an alert to explain", options,
            format_func=lambda i: f"{view.loc[i, 'type']} | amount {view.loc[i, 'amount']:,.0f} | score {view.loc[i, 'score']:.3f}",
        )
        col_a, col_b = st.columns(2)
        with col_a:
            st.markdown("**Why the Random Forest scored it this way (SHAP)**")
            shap_waterfall(scored.loc[[chosen], FEATURES])
        with col_b:
            st.markdown("**Account neighbourhood**")
            draw_neighbourhood(chosen, score_col)
    else:
        st.info("No transactions match the current filters.")

# ------------------------------------------------------------------ live scoring
with tab_score:
    st.subheader("Score a transaction (Random Forest)")
    with st.form("score_form"):
        a, b = st.columns(2)
        tx_type = a.selectbox("Transaction type", TYPES, index=TYPES.index("TRANSFER"))
        hour = b.slider("Hour of day", 0, 23, 12)
        amount = a.number_input("Amount", min_value=0.0, value=100000.0, step=1000.0)
        old_orig = b.number_input("Origin balance before", min_value=0.0, value=100000.0, step=1000.0)
        new_orig = a.number_input("Origin balance after", min_value=0.0, value=0.0, step=1000.0)
        old_dest = b.number_input("Destination balance before", min_value=0.0, value=0.0, step=1000.0)
        new_dest = a.number_input("Destination balance after", min_value=0.0, value=100000.0, step=1000.0)
        merchant = b.checkbox("Destination is a merchant")
        submitted = st.form_submit_button("Score transaction")

    if submitted:
        row = pd.DataFrame([{
            "step": hour, "type": tx_type, "amount": amount,
            "oldbalanceOrg": old_orig, "newbalanceOrig": new_orig,
            "oldbalanceDest": old_dest, "newbalanceDest": new_dest,
            "nameDest": "M0" if merchant else "C0",
        }])
        features = engineer_features(row)[FEATURES]
        probability = float(bundle["model"].predict_proba(features)[:, 1][0])
        tier = risk_tier(np.array([probability]), bundle["threshold"])[0]

        m1, m2 = st.columns(2)
        m1.metric("Fraud probability", f"{probability:.1%}")
        m2.metric("Risk tier", tier)
        if tier == "High":
            st.error("High risk: recommend manual review.")
        elif tier == "Medium":
            st.warning("Medium risk: monitor.")
        else:
            st.success("Low risk.")
        st.markdown("**Feature contributions (SHAP)**")
        shap_waterfall(features)


Launch the dashboard from a terminal in this folder:

```
streamlit run app.py
```

In [ ]:
# Compile-check the generated app so syntax errors surface here, not in the browser
compile(Path("app.py").read_text(encoding="utf-8"), "app.py", "exec")
print("app.py written and compiles.")
print("Run:  streamlit run app.py")


### 12.4 Limitations and Next Steps

- **Synthetic data:** PaySim fraud follows simulator rules. Strong scores do not imply real-world performance.
- **Post-transaction features:** if they are unavailable at scoring time, use the pre-transaction results as the realistic estimate.
- **Sparse graph:** accounts rarely repeat in PaySim, which limits what message passing can add. Datasets with rings, mule accounts or repeated counterparties (for example Elliptic or IBM AML) would exercise the GNN better.
- **Next steps:** neighbour-sampling for the full training graph, temporal graph models, cost-sensitive thresholds (cost of a missed fraud vs cost of a review), and drift monitoring in the dashboard.